In [1]:
from audio_processing import AudioSignal, TimeFeatures, STFTFeatures
import numpy as np 
import os

In [2]:
def safe_mean(x):
    return float(np.mean(x)) if np.size(x) else 0.0

In [7]:
audio_dir = 'D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/'
audio_sigs = []

for dirpath, dirnames, filenames in os.walk(audio_dir):
    for filename in filenames:
        if filename.endswith('.mp3') or filename.endswith('.wav'):
            # audio_paths.append(os.path.join(dirpath, filename))
            sig = AudioSignal(os.path.join(dirpath, filename))
            if not sig.invalid:
                audio_sigs.append((filename, sig))

d:\Engineering\Signal Processing\Personal Projects\Song Analysis\audio_processing.py:79: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=True)
c:\Users\mythk\anaconda3\envs\mir\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Clipse, John Legend - The Birds Don't Sing\Clipse, John Legend - The Birds Don't Sing.mp3
[WARN] Clipped / malformed audio: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Glorb - The Bottom 3\Glorb - The Bottom 3.mp3
[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Greentea Peng - Satta\Greentea Peng - Satta.mp3
[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Joyner Lucas, Ava Maxx - Tear Me Down\Joyner Lucas, Ava Maxx - Tear Me Down.mp3
[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Lady Gaga - Alejandro\Lady Gaga - Alejandro.mp3
[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Lemonade Mouth - Determin

In [4]:
def sec_to_samples(sec, sr):
    return int(round(sec*sr))

In [5]:
def make_windows(y, sr, window_sec=10, hop_sec=None):
    if hop_sec is None:
        hop_sec = window_sec # Non-overlapping by default

    win = sec_to_samples(window_sec, sr)
    hop = sec_to_samples(hop_sec, sr)

    windows = []
    for start in range(0, max(1, len(y) - win), hop):
        end = start + win
        windows.append(y[start:end])

    return windows

In [6]:
def analyze_window(y_seg, sr, N=2048, H=512):
    tmp = AudioSignal.__new__(AudioSignal)
    tmp.y = y_seg
    tmp.sr = sr
    tmp.N = N
    tmp.H = H
    tmp.invalid = (np.std(y_seg) < 1e-6)
    tmp._cache = {}
    
    return tmp

In [9]:
for siginfo in audio_sigs:
    filename, sig = siginfo
    sig_win_5s = make_windows(sig.y, sig.sr, window_sec=5, hop_sec=2)
    sig_win_10s = make_windows(sig.y, sig.sr, window_sec=10, hop_sec=5)

    results_5s = []
    results_10s = []

    for w in sig_win_5s:
        sig_win = analyze_window(w, sig.sr)
        tf = TimeFeatures(sig_win)
        sf = STFTFeatures(sig_win)

        results_5s.append({
            "loudness_time": tf.loudness_time(),
            "loudness_stft": sf.loudness_stft(),
            "speechiness_time": tf.speechiness_time(),
            "speechiness_stft": sf.speechiness_stft(),
            "acousticness_time": tf.acousticness_time(),
            "acousticness_stft": sf.acousticness_stft(),
            "liveness_time": tf.liveness_time(),
            "liveness_stft": sf.liveness_stft(),
            "instrumentalness_time": tf.instrumentalness_time(),
            "instrumentalness_stft": sf.instrumentalness_stft()
        })

    for w in sig_win_10s:
        sig_win = AudioSignal(w, sig.sr)
        tf = TimeFeatures(sig_win)
        sf = STFTFeatures(sig_win)

        results_10s.append({
            "energy_time": tf.energy_time(),
            "energy_stft": sf.energy_stft(),
            "danceability_time": tf.danceability_time(),
            "danceability_stft": sf.danceability_stft(),
            "valence_stft": sf.valence_stft(),
            "tempo_time": tf.tempo_time(),
            "tempo_stft": sf.tempo_stft(),
            "time_signature_time": tf.time_sig_time(),
            "time_signature_stft": sf.time_sig_freq()
        })

AttributeError: 'TimeFeatures' object has no attribute '_cache'

In [4]:
failed_files = []

for audio_file in audio_paths:
    print("Processing file:", audio_file)
    print("--------------------------------------------------")

    try:
        time_features = TimeFeatures(audio_file)
    except Exception as e:
        print(f"Error processing {audio_file}: {e}")
        failed_files.append(audio_file)
        continue

    if time_features.invalid:
        print(f"Skipping invalid file: {audio_file}")
        failed_files.append(audio_file)
        continue

    amplitude = time_features._amplitude()
    print("Amplitude:", np.mean(amplitude))
    amplitude_db = time_features._amplitude_dB()
    print("Amplitude (dB):", np.mean(amplitude_db))
    peak_amplitude = time_features._peak_amplitude()
    print("Peak Amplitude:", np.mean(peak_amplitude))
    peak_dB = time_features._peak_amplitude_dB()
    print("Peak Amplitude (dB):", np.mean(peak_dB))
    crest_factor = time_features._crest_factor()
    print("Crest Factor:", np.mean(crest_factor))
    crest_factor_dB = time_features._crest_factor_dB()
    print("Crest Factor (dB):", np.mean(crest_factor_dB))
    crest_factor_track = time_features._crest_factor_track()
    print("Crest Factor Track:", crest_factor_track)
    energy_envelope = time_features._energy_envelope()
    print("Energy Envelope:", np.mean(energy_envelope))
    energy_var = time_features._energy_variance()
    print("Energy Variance:", energy_var)
    energy_mod_rate = time_features._energy_mod_rate()
    print("Energy Modulation Rate:", energy_mod_rate)
    energy_modulation = time_features._energy_modulation_signal()
    print("Energy Modulation Signal:", np.mean(energy_modulation))
    zero_crossing_rate = time_features._zero_crossing_rate()
    print("Zero Crossing Rate:", np.mean(zero_crossing_rate))
    dynamic_range = time_features._dynamic_range()
    print("Dynamic Range:", dynamic_range)
    onset_env = time_features._onset_env()
    print("Onset Envelope:", safe_mean(onset_env))
    onset_autocorr = time_features._onset_autocorr()
    print("Onset Autocorrelation:", safe_mean(onset_autocorr))
    pulse_clarity = time_features._pulse_clarity()
    print("Pulse Clarity:", pulse_clarity)
    tempo_var = time_features._tempo_var()
    print("Tempo Variance:", tempo_var)
    silence_features = time_features._silence_ratio()
    print("Silence Ratio:", silence_features)
    print("--------------------------------------------------")

Processing file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Backstreet Boys - Not For Me\Backstreet Boys - Not For Me.mp3
--------------------------------------------------


d:\Engineering\Signal Processing\Personal Projects\Song Analysis\audio_processing.py:79: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=True)
c:\Users\mythk\anaconda3\envs\mir\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Amplitude: 0.19656938
Amplitude (dB): -16.548492
Peak Amplitude: 0.64352447
Peak Amplitude (dB): -6.1481833
Crest Factor: 3.5923822
Crest Factor (dB): 10.164426
Crest Factor Track: 3.084599256515503
Energy Envelope: 101.17875791224549
Energy Variance: 0.773949775233206
Energy Modulation Rate: 9.250672006559682
Energy Modulation Signal: 0.08034065
Zero Crossing Rate: 0.09663162512437626
Dynamic Range: 16.10682487487793
Onset Envelope: 0.06425538659095764
Onset Autocorrelation: 6.61995800328441e-05
Pulse Clarity: 1.0
Tempo Variance: (130.48634070288372, 98.74167777279804, 0.0)
Silence Ratio: 0.10003564215278603
--------------------------------------------------
Processing file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Bon Jovi - You Give Love a Bad Name\Bon Jovi - You Give Love a Bad Name.mp3
--------------------------------------------------


d:\Engineering\Signal Processing\Personal Projects\Song Analysis\audio_processing.py:79: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None, mono=True)
c:\Users\mythk\anaconda3\envs\mir\Lib\site-packages\librosa\core\audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Amplitude: 0.13226946
Amplitude (dB): -19.251263
Peak Amplitude: 0.44459033
Peak Amplitude (dB): -8.713381
Crest Factor: 3.3430796
Crest Factor (dB): 10.127882
Crest Factor Track: 3.2709591388702393
Energy Envelope: 39.52308911413926
Energy Variance: 0.29497485804496837
Energy Modulation Rate: 14.139376876319782
Energy Modulation Signal: 0.05070579
Zero Crossing Rate: 0.0974760470398498
Dynamic Range: 7.368486404418945
Onset Envelope: 0.07827688008546829
Onset Autocorrelation: 5.267831147648394e-05
Pulse Clarity: 1.0
Tempo Variance: (4.191603446687408, 123.67971171499318, 0.9661092074959999)
Silence Ratio: 0.09999480276492906
--------------------------------------------------
Processing file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Clipse, John Legend - The Birds Don't Sing\Clipse, John Legend - The Birds Don't Sing.mp3
--------------------------------------------------
[ERROR] Pure silence detected: D:/Engineering/Signal Processing/Personal Proje

In [5]:
for f in failed_files:
    print("Failed file:", f)

Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Clipse, John Legend - The Birds Don't Sing\Clipse, John Legend - The Birds Don't Sing.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Glorb - The Bottom 3\Glorb - The Bottom 3.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Greentea Peng - Satta\Greentea Peng - Satta.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Joyner Lucas, Ava Maxx - Tear Me Down\Joyner Lucas, Ava Maxx - Tear Me Down.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Lady Gaga - Alejandro\Lady Gaga - Alejandro.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Lemonade Mouth - Determinate\Lemonade Mouth - Determinate.mp3
Failed file: D:/Engineering/Signal Processing/Personal Projects/Song Analy

In [6]:
for audio_file in audio_paths:
    if audio_file in failed_files:
        continue

    print("Processing file:", audio_file)
    print("--------------------------------------------------")

    time_features = TimeFeatures(audio_file)

    loudness = time_features.loudness_time()
    print("Loudness Time:", loudness)
    energy = time_features.energy_time()
    print("Energy Time:", energy)
    speechiness = time_features.speechiness_time()
    print("Speechiness Time:", speechiness)
    acousticness = time_features.acousticness_time()
    print("Acousticness Time:", acousticness)
    danceability = time_features.danceability_time()
    print("Danceability Time:", danceability)
    tempo = time_features.tempo_time()
    print("Tempo Time:", tempo)
    liveness = time_features.liveness_time()
    print("Liveness Time:", liveness)
    instrumentalness = time_features.instrumentalness_time()
    print("Instrumentalness Time:", instrumentalness)
    time_signature = time_features.time_sig_time()
    print("Time Signature Time:", time_signature)
    print("--------------------------------------------------")

Processing file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Backstreet Boys - Not For Me\Backstreet Boys - Not For Me.mp3
--------------------------------------------------
Loudness Time: -9.814973831176758
Energy Time: 0.7364390845310685
Speechiness Time: 0.3842796665445679
Acousticness Time: 0.29212226767224536
Danceability Time: 0.6072942406700739
Tempo Time: 95.26867901984095
Liveness Time: 0.8074229997139393
Instrumentalness Time: 0.6475841353914314
Time Signature Time: 5
--------------------------------------------------
Processing file: D:/Engineering/Signal Processing/Personal Projects/Song Analysis/dataset/songs/Bon Jovi - You Give Love a Bad Name\Bon Jovi - You Give Love a Bad Name.mp3
--------------------------------------------------
Loudness Time: -15.033650207519532
Energy Time: 0.6673732782132573
Speechiness Time: 0.3976246218996504
Acousticness Time: 0.21171834194554873
Danceability Time: 0.8127280286887508
Tempo Time: 120.76043091000